# Annotation using Gemini
- Create "ground truth" OCR data using Gemini on the SROIE 2019 dataset.
- Used model: Gemini 2.0 Flash

In [ ]:
from dotenv import load_dotenv
from PIL import Image
from google import genai
import time
import glob
import os
model_name = 'gemini-2.0-flash'
api_sleep_time = 5 # In seconds.
load_dotenv()

## 1. Load dataset

In [ ]:
# Legacy credential-writing commands removed. Authenticate Kaggle externally.
!kaggle datasets download -d urbikn/sroie-datasetv2
!unzip -q sroie-datasetv2.zip -d "sroie_v2"

In [ ]:
image_paths_list = [
    './sroie_v2/SROIE2019/train/img/*.jpg', # Train
    './sroie_v2/SROIE2019/test/img/*.jpg' # Test
]

# Example
sample = Image.open('./sroie_v2/SROIE2019/train/img/X51005433552.jpg')
sample

## 2. Sample call to Gemini

In [ ]:
# Sample call
from google.colab import userdata
client = genai.Client(api_key=userdata.get('GEMINI_API'))
response = client.models.generate_content(
    model=model_name,
    contents=['OCR this image. Do not give any other content', sample]
)
print(response)
print('#'*50)
print(response.text)

## 3. Annotating

In [ ]:
def annotate_images(image_paths_list, out_dir_list, client, model_name, message, api_sleep_time=5, resize_dim=(1024, 1536)):
    for image_path_split, out_dir in zip(image_paths_list, out_dir_list):
        all_images = glob.glob(image_path_split)
        os.makedirs(out_dir, exist_ok=True)
        total_images = len(all_images)
        print(f"Total images in split: {total_images}")

        image_counter = 0
        while image_counter < total_images:
            print(f"Image {image_counter}: Image path: {all_images[image_counter]}")
            image = Image.open(all_images[image_counter])
            image = image.resize(resize_dim)

            response = client.models.generate_content(
                model=model_name,
                contents=[message, image]
            )
            response_text = response.text

            out_path = os.path.join(
                out_dir,
                all_images[image_counter].split(os.path.sep)[-1].split('.jpg')[0] + '.txt'
            )
            with open(out_path, 'w') as f:
                f.write(response_text)
            image_counter += 1
            print(f"Number of response characters returned: {len(response_text)}")
            print('-'*50)
            time.sleep(api_sleep_time)

# use
out_dir_list = [
    f"./annots/{model_name}/train_annots", #Train
    f"./annots/{model_name}/test_annots" # Test
]
message = 'Give the OCR text from this image and nothing else. Read the bill image carefully and extract all the information. Capture every possible detail including Restaurant name, Address, Bill number, order number, date, time, Itemized list, total bill amount, GST number, Website, email, and footer notes.'
annotate_images(image_paths_list, out_dir_list, client, model_name, message)